In [1]:
!pip install transformers datasets trl wandb ollama -q

In [2]:
import torch
print(torch.cuda.is_available())          # must be True
print(torch.cuda.get_device_name(0))      # should show GPU name
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

True
NVIDIA GeForce RTX 5070 Ti
16.58585088 GB


In [6]:
!ollama pull deepseek-r1:14b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 400 KB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 565 KB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 1.4 MB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 2.0 MB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 2.4 MB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 3.1 MB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 3.7 MB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% ▕                  ▏ 4.2 MB/9.0 GB                  pulling manifest 
pulling 6e9f90f02bb3:   0% 

In [34]:
import ollama
import json
import re
from datasets import load_dataset
from tqdm import tqdm
import time


In [4]:
import ollama
models = ollama.list()
print([m['model'] for m in models['models']])  # should show deepseek-r1:14b

['deepseek-r1:14b', 'llama3:latest']


In [5]:
from datasets import load_dataset
ds = load_dataset("EleutherAI/hendrycks_math", "algebra", split="train")


In [6]:
print(ds)
from collections import Counter
print(Counter(ds['level'])) 

Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 1744
})
Counter({'Level 5': 436, 'Level 4': 398, 'Level 3': 392, 'Level 2': 340, 'Level 1': 178})


In [7]:
ds = ds.filter(lambda x: x['level'] in ['Level 1', 'Level 2', 'Level 3'])
print(f"Total problems: {len(ds)}")

Total problems: 910


In [29]:
# prompt and extraction
SYSTEM_PROMPT = """You are a math reasoning assistant.
You MUST follow this exact format and no other:
<think>
step by step reasoning here
</think>
<answer>final answer only, no explanation</answer>

Do not write anything outside these tags."""

In [30]:
def extract_answer(text):
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    return match.group(1).strip() if match else None

In [31]:
def extract_boxed(text):
    """
    Extracts the final answer from the MATH dataset's gold solution.
    
    The MATH dataset always wraps the correct answer in LaTeX \\boxed{} notation
    e.g. '\\boxed{42}' or '\\boxed{\\frac{1}{2}}'.
    
    Also strips any \\text{} commands that sometimes appear inside the box
    e.g. '\\boxed{2 \\text{ euros}}' → '2'
    
    Args:
        text (str): Full gold solution string from the dataset
    Returns:
        str | None: Extracted answer, or None if no \\boxed{} found
    """
    match = re.search(r'\\boxed\{([^{}]+)\}', text)
    if match:
        clean = re.sub(r'\\text\{[^}]*\}', '', match.group(1))
        return clean.strip()
    return None


def extract_answer_tag(text):
    """
    Extracts the final answer from the model's generated output.
    
    Since the model doesn't always follow the <answer> tag format strictly,
    this function tries four fallback patterns in order:
        1. <answer>42</answer>     — ideal case, explicit tags
        2. **Answer:** 42          — bold markdown format
        3. \\boxed{42}             — model wrote LaTeX box itself
        4. Answer: 42              — plain text fallback
    
    Returns the first match found, or None if nothing matches.
    
    Args:
        text (str): Full model output string
    Returns:
        str | None: Extracted answer, or None if no pattern matched
    """
    # Format 1: <answer>...</answer>
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if match:
        return match.group(1).strip()

    # Format 2: **Answer:** 42
    match = re.search(r'\*\*Answer:\*\*\s*(.+)', text)
    if match:
        return match.group(1).strip()

    # Format 3: \boxed{} in model output
    match = re.search(r'\\boxed\{([^{}]+)\}', text)
    if match:
        return match.group(1).strip()

    # Format 4: "Answer: 42" plain text
    match = re.search(r'[Aa]nswer:\s*(.+)', text)
    if match:
        return match.group(1).strip()

    return None


def normalize(text):
    return (text
        .replace('$', '')
        .replace(' ', '')
        .replace('\n', '')
        .replace(r'\(', '')
        .replace(r'\)', '')
        .replace(r'\[', '')
        .replace(r'\]', '')
        .replace(r'\boxed{', '')
        .replace('}', '')
        .strip()
        .lower())

def generate_trace(problem):
    """
    Sends a math problem to DeepSeek-R1-14B running locally via Ollama
    and returns the full model response.
    
    The response contains the full reasoning trace inside <think> tags
    and the final answer inside <answer> tags, as instructed by SYSTEM_PROMPT.
    
    Args:
        problem (str): Math problem string from the dataset
    Returns:
        str: Full model response including reasoning trace and final answer
    """
    response = ollama.chat(
        model="deepseek-r1:14b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem}
        ],
        options={"num_predict": 2048}
    )
    return response['message']['content']

In [33]:
for i, item in enumerate(ds.select(range(10))):
    trace = generate_trace(item['problem'])
    predicted = extract_answer_tag(trace)
    gold_raw = item['solution']
    gold_boxed = extract_boxed(item['solution'])

    print(f"\n{'='*60}")
    print(f"[Problem {i+1}] Level: {item['level']}")
    print(f"Problem: {item['problem'][:100]}...")
    print(f"\n--- Gold (full solution) ---")
    print(gold_raw[:200])
    print(f"\n--- Gold (boxed extracted) ---")
    print(gold_boxed)
    print(f"\n--- Model trace ---")
    print(trace[:300])
    print(f"\n--- Model answer (extracted) ---")
    print(predicted)
    print(f"\n--- Match? ---")
    if predicted and gold_boxed:
        print(f"Raw:        '{predicted}' == '{gold_boxed}' → {predicted.strip() == gold_boxed.strip()}")
        print(f"Normalized: '{normalize(predicted)}' == '{normalize(gold_boxed)}' → {normalize(predicted) == normalize(gold_boxed)}")
    else:
        print(f"predicted={predicted}, gold_boxed={gold_boxed} — extraction failed")


[Problem 1] Level: Level 3
Problem: What is the degree of the polynomial $(4 +5x^3 +100 +2\pi x^4 + \sqrt{10}x^4 +9)$?...

--- Gold (full solution) ---
This polynomial is not written in standard form.  However, we don't need to write it in standard form, nor do we need to pay attention to the coefficients.  We just look for the exponents on $x$.  We 

--- Gold (boxed extracted) ---
4

--- Model trace ---
The given polynomial is \(4 + 5x^3 + 100 + 2\pi x^4 + \sqrt{10}x^4 + 9\).

To determine its degree:
1. Identify each term's degree:
   - \(4\) and \(100\), \(9\) are constants with degree 0.
   - \(5x^3\) has degree 3.
   - \(2\pi x^4\) and \(\sqrt{10}x^4\) both have degree 4.

The highest non-zero 

--- Model answer (extracted) ---
4

--- Match? ---
Raw:        '4' == '4' → True
Normalized: '4' == '4' → True

[Problem 2] Level: Level 3
Problem: Evaluate $\left\lceil3\left(6-\frac12\right)\right\rceil$....

--- Gold (full solution) ---
Firstly, $3\left(6-\frac12\right)=18-1-\frac12=17

In [36]:
# --- Storage ---
results = []
skipped = 0
failed_reasons = Counter()
total_tokens = 0
total_time = 0

# --- Main generation loop ---
for item in tqdm(ds, desc="Generating traces", unit="problem"):
    start_time = time.time()

    try:
        response = ollama.chat(
            model="deepseek-r1:14b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item['problem']}
            ],
            options={"num_predict": 2048}
        )

        elapsed = time.time() - start_time
        trace = response['message']['content']

        # Token tracking
        prompt_tokens = response.get('prompt_eval_count', 0)
        completion_tokens = response.get('eval_count', 0)
        tokens = prompt_tokens + completion_tokens
        tps = completion_tokens / elapsed if elapsed > 0 else 0

        total_tokens += tokens
        total_time += elapsed

        # Extract and compare answers
        predicted = extract_answer_tag(trace)
        gold = extract_boxed(item['solution'])

        if not predicted:
            skipped += 1
            failed_reasons['no_answer_extracted'] += 1
            continue

        if not gold:
            skipped += 1
            failed_reasons['no_gold_extracted'] += 1
            continue

        if normalize(predicted) == normalize(gold):
            results.append({
                "problem": item['problem'],
                "trace": trace,
                "answer": gold,
                "level": item['level'],
                "type": item['type'],
                "tokens": tokens,
                "elapsed_sec": round(elapsed, 2),
                "tokens_per_sec": round(tps, 1)
            })
        else:
            skipped += 1
            failed_reasons['answer_mismatch'] += 1

    except Exception as e:
        skipped += 1
        failed_reasons[f'error: {str(e)[:50]}'] += 1
        continue

    # Live stats every 50 problems
    if (len(results) + skipped) % 50 == 0:
        avg_tps = total_tokens / total_time if total_time > 0 else 0
        tqdm.write(f"   Kept: {len(results)} |  Skipped: {skipped} | Avg tok/s: {avg_tps:.1f}")

# --- Final summary ---
print("\n" + "="*50)
print(f"DONE")
print(f"  Kept:            {len(results)}")
print(f"  Skipped:         {skipped}")
print(f"  Keep rate:       {len(results) / len(ds) * 100:.1f}%")
print(f"  Avg tokens/sec:  {total_tokens / total_time:.1f}")
print(f"  Total time:      {total_time / 60:.1f} mins")
print(f"\nSkip reasons:")
for reason, count in failed_reasons.most_common():
    print(f"  {reason}: {count}")



Generating traces:   5%|▌         | 50/910 [07:16<2:23:05,  9.98s/problem]

   Kept: 26 |  Skipped: 24 | Avg tok/s: 85.5


Generating traces:  16%|█▋        | 150/910 [21:15<2:15:26, 10.69s/problem]

   Kept: 85 |  Skipped: 65 | Avg tok/s: 85.3


Generating traces:  27%|██▋       | 250/910 [35:41<1:29:10,  8.11s/problem]

...

   Kept: 506 |  Skipped: 294 | Avg tok/s: 85.8


Generating traces:  93%|█████████▎| 850/910 [1:58:29<06:20,  6.34s/problem]

   Kept: 542 |  Skipped: 308 | Avg tok/s: 85.8


Generating traces: 100%|██████████| 910/910 [2:07:05<00:00,  8.38s/problem]


DONE
  Kept:            577
  Skipped:         333
  Keep rate:       63.4%
  Avg tokens/sec:  85.9
  Total time:      127.1 mins

Skip reasons:
  no_answer_extracted: 192
  answer_mismatch: 82
  no_gold_extracted: 59


In [37]:
# --- Save to JSONL ---
import os
os.makedirs("data/raw", exist_ok=True)

with open("data/raw/traces.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"\nSaved {len(results)} traces to data/raw/traces.jsonl")


Saved 577 traces to data/raw/traces.jsonl
